<a href="https://colab.research.google.com/github/kushalshah0/Detecting-AI-Generated-Phishing-Emails-Using-BERT/blob/main/ai_generated_phishing_email_detection_FastAPI_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title Install Dependencies
!pip install -qqq fastapi uvicorn transformers torch tensorflow pickle-mixin shap
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

/usr/local/bin/cloudflared: Text file busy


In [ ]:
#@title Verify model paths
import os

SAMPLE_NAME = "sample1"

BASE_PATH = f"/content/drive/MyDrive/Detect_AI_Phishing_Project/{SAMPLE_NAME}"

paths = {
    "LSTM": f"{BASE_PATH}/lstm_model.pt",
    "GRU": f"{BASE_PATH}/gru_model.pt",
    "BERT": f"{BASE_PATH}/bert/final_model",
    "Tokenizer": f"{BASE_PATH}/rnn_tokenizer.pkl"
}

for k, v in paths.items():
    print(k, "✅" if os.path.exists(v) else "❌", v)


LSTM ✅ /content/drive/MyDrive/Detect_AI_Phishing_Project/sample1/lstm_model.pt
GRU ✅ /content/drive/MyDrive/Detect_AI_Phishing_Project/sample1/gru_model.pt
BERT ✅ /content/drive/MyDrive/Detect_AI_Phishing_Project/sample1/bert/final_model
Tokenizer ✅ /content/drive/MyDrive/Detect_AI_Phishing_Project/sample1/rnn_tokenizer.pkl


In [ ]:
#@title Create FastAPI project structure
import os

API_DIR = "/content/api"
os.makedirs(API_DIR, exist_ok=True)

files = ["main.py", "models.py", "schemas.py", "domain_trust.py", "url_features.py"]
for f in files:
    with open(os.path.join(API_DIR, f), "w") as fp:
        fp.write("")

print("FastAPI files created:", files)


FastAPI files created: ['main.py', 'models.py', 'schemas.py', 'domain_trust.py', 'url_features.py']


In [ ]:
#@title Write domain_trust.py
%%writefile /content/api/domain_trust.py
# domain_trust.py  -- Sender Domain Trust Scoring
import re
from typing import Optional

TRUSTED_DOMAINS: dict = {
    "google.com": 0.95, "googlemail.com": 0.95, "gmail.com": 0.75,
    "microsoft.com": 0.95, "outlook.com": 0.75, "hotmail.com": 0.70, "live.com": 0.70,
    "apple.com": 0.95, "icloud.com": 0.80,
    "amazon.com": 0.90, "amazon.co.uk": 0.90, "amazonaws.com": 0.85,
    "meta.com": 0.85, "facebook.com": 0.80, "instagram.com": 0.80,
    "twitter.com": 0.80, "x.com": 0.80, "linkedin.com": 0.85,
    "github.com": 0.90, "gitlab.com": 0.85, "dropbox.com": 0.85,
    "slack.com": 0.85, "zoom.us": 0.85, "adobe.com": 0.85,
    "netflix.com": 0.85, "spotify.com": 0.85, "stripe.com": 0.90,
    "paypal.com": 0.85, "ebay.com": 0.80,
    "chase.com": 0.85, "bankofamerica.com": 0.85, "wellsfargo.com": 0.85,
    "citibank.com": 0.85, "hsbc.com": 0.85, "barclays.com": 0.85,
    "cloudflare.com": 0.90, "digitalocean.com": 0.88, "vercel.com": 0.88, "netlify.com": 0.88,
}

TRUSTED_TLDS: dict = {
    ".gov": 0.92, ".mil": 0.92, ".edu": 0.80, ".ac.uk": 0.80, ".gov.uk": 0.90,
}

SUSPICIOUS_PATTERNS = [
    r"\.tk$", r"\.ml$", r"\.ga$", r"\.cf$", r"\.gq$",
    r"\.xyz$", r"\.top$", r"\.click$", r"\.loan$",
    r"\d{4,}",
    r"(paypa1|g00gle|amaz0n|micros0ft|app1e)",
    r"(login|signin|verify|secure|update|alert)\.",
]

def extract_domain(sender: str) -> Optional[str]:
    if not sender:
        return None
    sender = sender.strip()
    angle = re.search(r"<([^>]+)>", sender)
    if angle:
        sender = angle.group(1).strip()
    domain = sender.split("@", 1)[-1].lower().strip() if "@" in sender else sender.lower().strip()
    if "." not in domain:
        return None
    return domain.split("/")[0].split(":")[0] or None

def _label(s: float) -> str:
    return "High" if s >= 0.85 else ("Medium" if s >= 0.65 else "Low")

def get_domain_trust(sender: Optional[str]) -> dict:
    result = {"sender": sender, "domain": None, "trust_score": 0.0,
              "trust_label": "Unknown", "is_known": False, "is_suspicious": False}
    if not sender:
        return result
    domain = extract_domain(sender)
    result["domain"] = domain
    if not domain:
        return result
    for pat in SUSPICIOUS_PATTERNS:
        if re.search(pat, domain, re.IGNORECASE):
            result.update({"trust_score": 0.05, "trust_label": "Low", "is_suspicious": True})
            return result
    if domain in TRUSTED_DOMAINS:
        s = TRUSTED_DOMAINS[domain]
        result.update({"trust_score": s, "is_known": True, "trust_label": _label(s)})
        return result
    parts = domain.split(".")
    for i in range(1, len(parts)):
        parent = ".".join(parts[i:])
        if parent in TRUSTED_DOMAINS:
            s = round(TRUSTED_DOMAINS[parent] * 0.95, 4)
            result.update({"trust_score": s, "is_known": True, "trust_label": _label(s)})
            return result
    for tld, s in TRUSTED_TLDS.items():
        if domain.endswith(tld):
            result.update({"trust_score": s, "is_known": True, "trust_label": _label(s)})
            return result
    return result

def adjust_confidence_for_domain(label: str, confidence: float, sender: Optional[str], domain_weight: float = 0.35) -> tuple:
    info = get_domain_trust(sender)
    trust_score = info["trust_score"]
    if not info["is_known"] and not info["is_suspicious"]:
        info.update({"adjustment_applied": False, "domain_weight_used": 0.0})
        return label, confidence, info
    phishing_prob = confidence if label == "Phishing" else (1.0 - confidence)
    domain_phishing_prob = 1.0 - trust_score
    adj = phishing_prob * (1.0 - domain_weight) + domain_phishing_prob * domain_weight
    new_label = "Phishing" if adj >= 0.5 else "Legitimate"
    new_conf = adj if new_label == "Phishing" else (1.0 - adj)
    info.update({
        "adjustment_applied": True, "domain_weight_used": domain_weight,
        "original_label": label, "original_confidence": round(confidence, 4),
        "adjusted_phishing_prob": round(adj, 4),
    })
    return new_label, round(new_conf, 4), info


Overwriting /content/api/domain_trust.py


In [ ]:
#@title Write url_features.py
%%writefile /content/api/url_features.py
# url_features.py  -- URL Extraction & Trust Analysis
#
# WHY this matters:
#   BERT/LSTM/GRU see URLs as plain tokens and CANNOT distinguish:
#       support.google.com   (trusted)  vs  paypal-verify.tk  (phishing)
#   This module extracts every URL from the body, scores each domain
#   using the same trust registry as domain_trust.py, and produces a
#   url_trust_score (0=all suspicious, 1=all trusted) that is blended
#   with the model confidence inside main.py.

import re
from urllib.parse import urlparse
from domain_trust import get_domain_trust

URL_SHORTENERS = {
    "bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly",
    "short.io", "rb.gy", "cutt.ly", "is.gd", "buff.ly",
    "tiny.cc", "lnkd.in", "youtu.be", "amzn.to",
}

IMPERSONATED_BRANDS = [
    "google", "paypal", "microsoft", "apple", "amazon",
    "facebook", "netflix", "instagram", "whatsapp", "twitter",
    "linkedin", "dropbox", "chase", "wellsfargo", "bankofamerica",
]

def extract_urls(text: str) -> list:
    # Full URLs with scheme (https://...)
    http_urls = re.findall(r'https?://[^\s<>"{}|\\^`\[\]]+', text)

    # Bare domain links: support.google.com/answer/123  (no http prefix)
    bare_urls = re.findall(
        r'\b(?:[a-zA-Z0-9-]+\.)+(?:com|org|net|edu|gov|io|co|uk|de|fr'
        r'|jp|tk|ml|ga|cf|gq|xyz|top|click|loan|info|biz)'
        r'(?:/[^\s,;]*)?',
        text
    )

    # Deduplicate: skip bare entries whose domain is already in http_urls
    http_domains = set()
    for u in http_urls:
        try:
            http_domains.add(urlparse(u).netloc.lower())
        except Exception:
            pass

    filtered_bare = [b for b in bare_urls if b.split("/")[0].lower() not in http_domains]

    seen, unique = set(), []
    for u in http_urls + filtered_bare:
        if u not in seen:
            seen.add(u)
            unique.append(u)
    return unique


def _get_url_domain(url: str) -> str:
    try:
        if url.startswith("http"):
            d = urlparse(url).netloc.lower()
        else:
            d = url.split("/")[0].lower()
        return d.split(":")[0].strip()
    except Exception:
        return ""


def analyze_urls(text: str) -> dict:
    urls = extract_urls(text)
    trusted, suspicious, unknown = [], [], []
    ip_urls, shorteners, misleading = [], [], []

    for url in urls:
        domain = _get_url_domain(url)
        if not domain:
            continue

        # IP address URL (e.g. http://192.168.1.1/login)
        if re.match(r'^\d{1,3}(\.\d{1,3}){3}$', domain):
            ip_urls.append(url)
            suspicious.append(url)
            continue

        # URL shortener
        if domain in URL_SHORTENERS:
            shorteners.append(url)
            suspicious.append(url)
            continue

        trust_info = get_domain_trust(domain)

        # Misleading: brand keyword inside suspicious domain (google.evil.tk)
        is_misleading = False
        for brand in IMPERSONATED_BRANDS:
            if brand in domain and trust_info["trust_score"] < 0.5:
                misleading.append(url)
                suspicious.append(url)
                is_misleading = True
                break

        if is_misleading:
            continue

        if trust_info["trust_score"] >= 0.70:
            trusted.append(url)
        elif trust_info["is_suspicious"]:
            suspicious.append(url)
        else:
            unknown.append(url)

    # Overall URL trust score
    total = max(len(urls), 1)
    if suspicious:
        url_trust = max(0.05, 0.5 - (len(suspicious) / total) * 0.45)
    elif trusted:
        url_trust = min(0.95, 0.5 + (len(trusted) / total) * 0.40)
    else:
        url_trust = 0.5  # only unknown URLs -- neutral

    return {
        "url_count":          len(urls),
        "urls_found":         urls[:10],
        "trusted_count":      len(trusted),
        "suspicious_count":   len(suspicious),
        "unknown_count":      len(unknown),
        "has_ip_url":         len(ip_urls)    > 0,
        "has_url_shortener":  len(shorteners) > 0,
        "has_misleading_url": len(misleading) > 0,
        "trusted_urls":       trusted[:5],
        "suspicious_urls":    suspicious[:5],
        "unknown_urls":       unknown[:5],
        "ip_urls":            ip_urls[:3],
        "shortener_urls":     shorteners[:3],
        "misleading_urls":    misleading[:3],
        "url_trust_score":    round(url_trust, 4),
    }


def adjust_confidence_for_urls(
    label: str, confidence: float, url_features: dict, url_weight: float = 0.25,
) -> tuple:
    # url_weight=0.25: URLs contribute 25% to the final decision
    # Returns (new_label, new_confidence, adjustment_applied: bool)
    if url_features["url_count"] == 0:
        return label, confidence, False

    url_phishing_prob = 1.0 - url_features["url_trust_score"]
    phishing_prob     = confidence if label == "Phishing" else (1.0 - confidence)
    adjusted          = phishing_prob * (1.0 - url_weight) + url_phishing_prob * url_weight

    new_label = "Phishing" if adjusted >= 0.5 else "Legitimate"
    new_conf  = adjusted if new_label == "Phishing" else (1.0 - adjusted)
    return new_label, round(new_conf, 4), True


Overwriting /content/api/url_features.py


In [ ]:
#@title Write schemas.py
%%writefile /content/api/schemas.py
from pydantic import BaseModel, Field
from typing import List, Optional


class TokenScore(BaseModel):
    token: str
    shap_score: float


class EmailRequest(BaseModel):
    text:   str
    model:  str           # bert | lstm | gru
    sender: Optional[str] = Field(
        default=None,
        description="Sender e.g. 'noreply@google.com' -- enables domain-trust calibration.",
    )


class URLAnalysis(BaseModel):
    url_count:              int
    trusted_count:          int
    suspicious_count:       int
    unknown_count:          int
    has_ip_url:             bool
    has_url_shortener:      bool
    has_misleading_url:     bool
    trusted_urls:           List[str] = []
    suspicious_urls:        List[str] = []
    url_trust_score:        float
    url_adjustment_applied: bool = False


class DomainTrustInfo(BaseModel):
    sender:                 Optional[str]   = None
    domain:                 Optional[str]   = None
    trust_score:            float           = 0.0
    trust_label:            str             = "Unknown"
    is_known:               bool            = False
    is_suspicious:          bool            = False
    adjustment_applied:     bool            = False
    domain_weight_used:     float           = 0.0
    original_label:         Optional[str]   = None
    original_confidence:    Optional[float] = None
    adjusted_phishing_prob: Optional[float] = None


class PredictionResponse(BaseModel):
    model:        str
    prediction:   str
    confidence:   float
    top_tokens:   Optional[List[TokenScore]]  = None
    calibrated:   bool                        = Field(default=False)
    url_analysis: Optional[URLAnalysis]       = Field(default=None)
    domain_trust: Optional[DomainTrustInfo]   = Field(default=None)


Overwriting /content/api/schemas.py


In [ ]:
#@title Write models.py
%%writefile /content/api/models.py
# models.py
import torch
import torch.nn as nn
import pickle
import numpy as np
import shap
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
import threading

SAMPLE_NAME = "sample1"
BASE_PATH   = f"/content/drive/MyDrive/Detect_AI_Phishing_Project/{SAMPLE_NAME}"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Changed to use GPU if available

# ── Temperature Scaling ───────────────────────────────────────────────────────
# Divides raw logits by T before softmax/sigmoid to reduce overconfidence.
# T=1.0 no change | T=1.5 mild softening (default) | T=2.0 strong softening
# To tune: sweep T on a validation set and minimise ECE (Expected Calibration Error)
BERT_TEMPERATURE: float = 1.4
RNN_TEMPERATURE:  float = 1.5

# ── SHAP Dynamic Threshold ────────────────────────────────────────────────────
# threshold = max(SHAP_FLOOR, 75th-percentile of |scores|)
# Keeps only top-25% most impactful tokens instead of everything above 0.001
SHAP_FLOOR: float = 0.001


# ── RNN Architectures ─────────────────────────────────────────────────────────

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc   = nn.Linear(hidden_dim, output_dim)
    def forward(self, text):
        embedded = self.embedding(text)
        _, (hidden, _) = self.lstm(embedded)
        return self.fc(hidden.squeeze(0))


class GRUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc  = nn.Linear(hidden_dim, output_dim)
    def forward(self, text):
        embedded = self.embedding(text)
        _, hidden = self.gru(embedded)
        return self.fc(hidden.squeeze(0))


# ── Load Tokenizer & RNN Models ───────────────────────────────────────────────

with open(f"{BASE_PATH}/rnn_tokenizer.pkl", "rb") as f:
    rnn_tokenizer = pickle.load(f)

MAX_LEN          = 200
MODEL_VOCAB_SIZE = 20000
EMBEDDING_DIM    = 128
HIDDEN_DIM       = 128
OUTPUT_DIM       = 1

lstm_model = LSTMModel(MODEL_VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM).to(DEVICE)
lstm_model.load_state_dict(torch.load(f"{BASE_PATH}/lstm_model.pt", map_location=DEVICE))
lstm_model.eval()

gru_model = GRUModel(MODEL_VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM).to(DEVICE)
gru_model.load_state_dict(torch.load(f"{BASE_PATH}/gru_model.pt", map_location=DEVICE))
gru_model.eval()

# ── Load BERT ─────────────────────────────────────────────────────────────────

bert_tokenizer = BertTokenizer.from_pretrained(f"{BASE_PATH}/bert/final_model")
bert_model     = BertForSequenceClassification.from_pretrained(
    f"{BASE_PATH}/bert/final_model"
).to(DEVICE)
bert_model.eval()

# Determine the device for the Hugging Face pipeline based on the global DEVICE setting
pipeline_device = 0 if torch.cuda.is_available() else -1 # Revert to using GPU if available

bert_pipeline = pipeline(
    "text-classification",
    model=bert_model, tokenizer=bert_tokenizer,
    device=pipeline_device, # Use the determined device
    top_k=None, truncation=True, max_length=512,
)

bert_shap_explainer = shap.Explainer(bert_pipeline)
bert_lock           = threading.Lock()


# ── Prediction helpers ────────────────────────────────────────────────────────

def preprocess_rnn(text: str) -> torch.Tensor:
    seq           = rnn_tokenizer.texts_to_sequences([text])
    processed_seq = [[tid if tid < MODEL_VOCAB_SIZE else 0 for tid in s] for s in seq]
    padded        = np.zeros((1, MAX_LEN))
    padded[0, :min(MAX_LEN, len(processed_seq[0]))] = processed_seq[0][:MAX_LEN]
    return torch.tensor(padded, dtype=torch.long).to(DEVICE)


def predict_rnn(model: nn.Module, text: str):
    # Logit divided by RNN_TEMPERATURE before sigmoid to soften confidence
    with torch.no_grad():
        x      = preprocess_rnn(text)
        output = model(x)
        prob   = torch.sigmoid(output / RNN_TEMPERATURE).item()
        label  = "Phishing" if prob >= 0.5 else "Legitimate"
        conf   = prob if prob >= 0.5 else 1.0 - prob
        return label, conf


def predict_bert(text: str):
    # Logits divided by BERT_TEMPERATURE before softmax to soften confidence
    with bert_lock:
        inputs = bert_tokenizer(
            text, return_tensors="pt",
            truncation=True, padding=True, max_length=512
        ).to(DEVICE)
        with torch.no_grad():
            outputs       = bert_model(**inputs)
            scaled_logits = outputs.logits / BERT_TEMPERATURE
            probs         = torch.softmax(scaled_logits, dim=1)
            conf, pred    = torch.max(probs, dim=1)
            label         = "Phishing" if pred.item() == 1 else "Legitimate"
            return label, conf.item()


# ── Dynamic SHAP threshold ────────────────────────────────────────────────────

def _dynamic_shap_threshold(token_scores: list) -> float:
    # 75th-percentile of absolute scores, floored at SHAP_FLOOR
    # Keeps only top-25% most impactful tokens
    if not token_scores:
        return SHAP_FLOOR
    abs_vals = [abs(t["shap_score"]) for t in token_scores]
    return max(SHAP_FLOOR, float(np.percentile(abs_vals, 75)))


def explain_bert(text: str, top_n: int = None) -> list:
    with bert_lock:
        shap_values = bert_shap_explainer([text])
    phishing_idx = 1
    tokens       = shap_values.data[0]
    scores       = shap_values.values[0][:, phishing_idx]
    merged = {}
    for token, score in zip(tokens, scores.tolist()):
        key = token.strip().lower()
        if not key or key in ("[CLS]", "[SEP]", "[PAD]"):
            continue
        if all(c in '.,!?;:()[]{}|"\'/\-' for c in key):
            continue
        if key in merged:
            merged[key]["shap_score"] += score
        else:
            merged[key] = {"token": token.strip(), "shap_score": score}
    token_scores = sorted(merged.values(), key=lambda x: abs(x["shap_score"]), reverse=True)
    threshold    = _dynamic_shap_threshold(token_scores)
    token_scores = [t for t in token_scores if abs(t["shap_score"]) >= threshold]
    if top_n:
        token_scores = token_scores[:top_n]
    return [{"token": d["token"], "shap_score": round(d["shap_score"], 4)} for d in token_scores]


def _predict_rnn_proba_for_shap(model: nn.Module, inputs_array: np.ndarray) -> np.ndarray:
    tensor = torch.tensor(inputs_array, dtype=torch.long).to(DEVICE)
    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs   = torch.sigmoid(outputs / RNN_TEMPERATURE).cpu().numpy().reshape(-1, 1)
    return probs


rnn_background_text = [
    "Hello, how are you today?", "This is a test email.",
    "Please confirm your account details.", "Click here for more information.",
    "Thank you for your time.",
]
rnn_background_sequences = [
    [tid if tid < MODEL_VOCAB_SIZE else 0 for tid in s]
    for s in rnn_tokenizer.texts_to_sequences(rnn_background_text)
]
rnn_background_padded = np.zeros((len(rnn_background_sequences), MAX_LEN), dtype=int)
for i, seq in enumerate(rnn_background_sequences):
    rnn_background_padded[i, :min(MAX_LEN, len(seq))] = seq[:MAX_LEN]

lstm_shap_explainer = shap.KernelExplainer(
    lambda x: _predict_rnn_proba_for_shap(lstm_model, x), rnn_background_padded)
gru_shap_explainer = shap.KernelExplainer(
    lambda x: _predict_rnn_proba_for_shap(gru_model, x), rnn_background_padded)


def explain_rnn(model_explainer, rnn_tokenizer_obj, text: str, top_n: int = None) -> list:
    raw_ids = rnn_tokenizer_obj.texts_to_sequences([text])[0]
    padded  = np.zeros((1, MAX_LEN), dtype=int)
    padded[0, :min(MAX_LEN, len(raw_ids))] = raw_ids[:MAX_LEN]
    shap_vals = model_explainer.shap_values(padded)[0]
    token_score_list = []
    for i, token_id in enumerate(raw_ids):
        if i >= MAX_LEN:
            break
        word  = rnn_tokenizer_obj.index_word.get(token_id, f"[UNK_{token_id}]")
        score = shap_vals[i]
        if isinstance(score, np.ndarray):
            score = score.item()
        token_score_list.append({"token": word, "shap_score": float(score)})
    cleaned = [t for t in token_score_list
               if not all(c in '.,!?;:()[]{}|"\'/\-' for c in t["token"].strip())]
    sorted_scores = sorted(cleaned, key=lambda x: abs(x["shap_score"]), reverse=True)
    threshold     = _dynamic_shap_threshold(sorted_scores)
    sorted_scores = [t for t in sorted_scores if abs(t["shap_score"]) >= threshold]
    if top_n:
        sorted_scores = sorted_scores[:top_n]
    return [{"token": d["token"], "shap_score": round(d["shap_score"], 4)} for d in sorted_scores]

Overwriting /content/api/models.py


In [ ]:
#@title Write main.py
%%writefile /content/api/main.py
import sys
sys.path.insert(0, '/content/api')

from fastapi import FastAPI, HTTPException
from schemas import EmailRequest, PredictionResponse, DomainTrustInfo, URLAnalysis
from models import (
    predict_rnn, predict_bert, lstm_model, gru_model,
    explain_bert, lstm_shap_explainer, gru_shap_explainer,
    rnn_tokenizer, explain_rnn,
)
from domain_trust import adjust_confidence_for_domain
from url_features  import analyze_urls, adjust_confidence_for_urls

from typing import Optional, List
import traceback, time

app = FastAPI(
    title="AI-Generated Phishing Detection API",
    version="2.0",
)

ERROR_LOG_FILE = "/content/api/error_trace.log"


@app.post("/predict", response_model=PredictionResponse)
def predict(request: EmailRequest):
    """
    3-stage pipeline
    ----------------
    Stage 1 -- ML inference  (BERT/LSTM/GRU with temperature scaling)
    Stage 2 -- URL analysis  (trusted vs suspicious links inside body)
    Stage 3 -- Domain trust  (sender domain reputation, only if sender= given)

    Each stage blends its signal via weighted linear combination so every
    adjustment is fully transparent in the response JSON.
    """
    text       = request.text
    model_name = request.model.lower()
    sender     = request.sender

    label:      str
    confidence: float
    top_tokens: Optional[List] = None
    url_info                   = None
    domain_trust_info          = None

    try:
        # ── Stage 1: ML model inference ───────────────────────────────────────
        if model_name == "bert":
            label, confidence = predict_bert(text)
            top_tokens        = explain_bert(text) if label == "Phishing" else None
        elif model_name == "lstm":
            label, confidence = predict_rnn(lstm_model, text)
            top_tokens = explain_rnn(lstm_shap_explainer, rnn_tokenizer, text) if label == "Phishing" else None
        elif model_name == "gru":
            label, confidence = predict_rnn(gru_model, text)
            top_tokens = explain_rnn(gru_shap_explainer, rnn_tokenizer, text) if label == "Phishing" else None
        else:
            raise HTTPException(status_code=400, detail="Invalid model. Choose from: bert, lstm, gru")

        # ── Stage 2: URL analysis & confidence adjustment ─────────────────────
        url_features = analyze_urls(text)
        label, confidence, url_adjusted = adjust_confidence_for_urls(
            label, confidence, url_features, url_weight=0.25
        )
        url_info = URLAnalysis(
            url_count              = url_features["url_count"],
            trusted_count          = url_features["trusted_count"],
            suspicious_count       = url_features["suspicious_count"],
            unknown_count          = url_features["unknown_count"],
            has_ip_url             = url_features["has_ip_url"],
            has_url_shortener      = url_features["has_url_shortener"],
            has_misleading_url     = url_features["has_misleading_url"],
            trusted_urls           = url_features["trusted_urls"],
            suspicious_urls        = url_features["suspicious_urls"],
            url_trust_score        = url_features["url_trust_score"],
            url_adjustment_applied = url_adjusted,
        )
        if label == "Legitimate":
            top_tokens = None

        # ── Stage 3: Sender domain trust (only when sender is provided) ───────
        if sender:
            label, confidence, raw_info = adjust_confidence_for_domain(
                label, confidence, sender, domain_weight=0.35
            )
            domain_trust_info = DomainTrustInfo(**raw_info)
            if label == "Legitimate":
                top_tokens = None

        return PredictionResponse(
            model        = model_name,
            prediction   = label,
            confidence   = round(confidence, 4),
            top_tokens   = top_tokens,
            calibrated   = True,
            url_analysis = url_info,
            domain_trust = domain_trust_info,
        )

    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc(file=sys.stderr)
        with open(ERROR_LOG_FILE, "a") as f:
            f.write(f"[ERROR - {time.ctime()}]\n")
            traceback.print_exc(file=f)
            f.write("\n" + "=" * 80 + "\n")
        raise HTTPException(status_code=500, detail=f"Internal Server Error: {e}")


Overwriting /content/api/main.py


In [ ]:
#@title Initialize FastAPI
import subprocess
import time
import os

# Kill any processes running on port 8000 or any uvicorn process
!pkill -f uvicorn || true
!fuser -k 8000/tcp || true

%cd /content/api

# Clear error log file before starting server
with open("error_trace.log", "w") as f:
    f.write("")

# Open nohup.out in write mode to clear it for new logs
with open("nohup.out", "w") as f:
    f.write("") # Clear the file

# Run uvicorn as a detached background process using subprocess.Popen
# Redirect stdout and stderr to nohup.out, with debug log level
uvicorn_process = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "debug"],
    stdout=open("nohup.out", "a"), # Append to log file
    stderr=subprocess.STDOUT,      # Redirect stderr to stdout
    preexec_fn=os.setsid           # Detach from parent process group
)

print("FastAPI server starting...")

server_ready = False
for _ in range(60): # Wait up to 60 seconds (60 * 1 second checks)
    time.sleep(1)
    if os.path.exists("nohup.out"):
        with open("nohup.out", "r") as f:
            log_content = f.read()
            if "Uvicorn running on http://0.0.0.0:8000" in log_content:
                server_ready = True
                break
    # Check if the process has terminated early
    if uvicorn_process.poll() is not None:
        print(f"FastAPI server terminated unexpectedly with exit code {uvicorn_process.returncode} during startup.")
        print("Check nohup.out for details.")
        break

if server_ready:
    print("FastAPI server is running and listening on port 8000.")
else:
    print("❌ FastAPI server did not start successfully or bind to port 8000 within the timeout.")
    print("Checking final nohup.out for details...")
    if os.path.exists("nohup.out"):
        with open("nohup.out", "r") as f:
            print(f.read())
    else:
        print("nohup.out not found.")

^C
/content/api
FastAPI server starting...
FastAPI server is running and listening on port 8000.


In [ ]:
#@title Display FastAPI Server Logs
import os

!cat nohup.out


/content/api/models.py:153: SyntaxWarning: invalid escape sequence '\-'
  if all(c in '.,!?;:()[]{}|"\'/\-' for c in key):
/content/api/models.py:210: SyntaxWarning: invalid escape sequence '\-'
  if not all(c in '.,!?;:()[]{}|"\'/\-' for c in t["token"].strip())]
2026-03-28 17:19:55.246495: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774718395.325364    6849 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774718395.352247    6849 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774718395.444412    6849 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than onc

In [ ]:
#@title Test API Locally (v2.0 -- URL analysis + domain trust + calibrated confidence)
import requests, json

API_URL = "http://127.0.0.1:8000/predict"

email_text = "Dear user, Our security systems have detected unusual login activity on your account.  To protect your account, you must verify your credentials within 24 hours  or your access will be temporarily suspended.  Please click the link below to confirm your identity:  https://phishing.test/verify  If you do not verify within 24 hours, your account will be locked pending a  security review.  IT Security Team  ABC Securities" #@param {type:"string"}

model_selector = "bert" #@param ["bert", "lstm", "gru"]
sender_input   = "" #@param {type:"string"}

payload = {
    "text":   email_text,
    "model":  model_selector,
    "sender": sender_input.strip() if sender_input.strip() else None,
}

response = requests.post(API_URL, json=payload)

if response.ok: # Check if the request was successful
    result   = response.json()

    SEP = "-" * 52
    print("\n" + "=" * 52)
    print(f"  PHISHING DETECTION  (model={result['model']})")
    print("=" * 52)
    icon = "PHISHING" if result["prediction"] == "Phishing" else "LEGITIMATE"
    print(f"  Result     : {icon}")
    print(f"  Confidence : {result['confidence']:.1%}  (calibrated={result.get('calibrated')})")

    if result.get("url_analysis"):
        ua = result["url_analysis"]
        print(f"\n  {SEP}")
        print(f"  URL ANALYSIS  ({ua['url_count']} links found in body)")
        print(f"  {SEP}")
        print(f"  Trusted    : {ua['trusted_count']}   {ua['trusted_urls']}")
        print(f"  Suspicious : {ua['suspicious_count']}   {ua['suspicious_urls']}")
        print(f"  Unknown    : {ua['unknown_count']}")
        print(f"  IP URL     : {ua['has_ip_url']}")
        print(f"  Shortener  : {ua['has_url_shortener']}")
        print(f"  Misleading : {ua['has_misleading_url']}")
        score = ua['url_trust_score']
        bar   = "#" * int(score * 20)
        print(f"  URL Trust  : {score:.2f}  [{bar:<20}]  (0=suspicious, 1=trusted)")
        print(f"  Adjusted   : {ua['url_adjustment_applied']}")

    if result.get("domain_trust"):
        dt = result["domain_trust"]
        print(f"\n  {SEP}")
        print(f"  SENDER DOMAIN TRUST")
        print(f"  {SEP}")
        print(f"  Domain      : {dt['domain']}")
        print(f"  Trust Score : {dt['trust_score']}  ({dt['trust_label']})")
        if dt.get("adjustment_applied"):
            print(f"  Before adj  : {dt['original_label']} @ {dt['original_confidence']:.1%}")
            print(f"  After adj   : {result['prediction']} @ {result['confidence']:.1%}")

    if result.get("top_tokens"):
        print(f"\n  {SEP}")
        print(f"  TOP SHAP TOKENS  ({len(result['top_tokens'])} tokens -- dynamic top-25% threshold)")
        print(f"  {SEP}")
        for t in result["top_tokens"]:
            direction = "PHISHING signal" if t["shap_score"] > 0 else "Legit signal"
            bar = "#" * min(20, int(abs(t["shap_score"]) * 800))
            print(f"  {t['token']:<15} {t['shap_score']:+.4f}  {bar}  ({direction})")
    elif result["prediction"] == "Legitimate":
        print(f"\n  No SHAP tokens -- email classified as Legitimate.")

    print("\n" + "=" * 52)
    print("\nFull JSON:\n", json.dumps(result, indent=2))

else:
    print(f"❌ API request failed with status code {response.status_code}")
    print(f"Error details: {response.text}")
    print("Please ensure the FastAPI server is running correctly and accessible.")



  PHISHING DETECTION  (model=bert)
  Result     : PHISHING
  Confidence : 87.0%  (calibrated=True)

  ----------------------------------------------------
  URL ANALYSIS  (1 links found in body)
  ----------------------------------------------------
  Trusted    : 0   []
  Suspicious : 0   []
  Unknown    : 1
  IP URL     : False
  Shortener  : False
  Misleading : False
  URL Trust  : 0.50  [##########          ]  (0=suspicious, 1=trusted)
  Adjusted   : True

  ----------------------------------------------------
  TOP SHAP TOKENS  (13 tokens -- dynamic top-25% threshold)
  ----------------------------------------------------
  Securities      +0.1278  ####################  (PHISHING signal)
  Dear            +0.1003  ####################  (PHISHING signal)
  Team            -0.0860  ####################  (Legit signal)
  https           +0.0831  ####################  (PHISHING signal)
  your            +0.0777  ####################  (PHISHING signal)
  account         +0.0676  ####

In [ ]:
#@title Display Specific Error Log
import os

ERROR_LOG_FILE = "/content/api/error_trace.log"

if os.path.exists(ERROR_LOG_FILE) and os.path.getsize(ERROR_LOG_FILE) > 0:
    print(f"--- Contents of {ERROR_LOG_FILE} ---")
    with open(ERROR_LOG_FILE, "r") as f:
        print(f.read())
    print("------------------------------------")
else:
    print(f"No specific error trace found in {ERROR_LOG_FILE}.")
    print("Checking general nohup.out for clues...")
    !cat nohup.out


No specific error trace found in /content/api/error_trace.log.
Checking general nohup.out for clues...
/content/api/models.py:153: SyntaxWarning: invalid escape sequence '\-'
  if all(c in '.,!?;:()[]{}|"\'/\-' for c in key):
/content/api/models.py:210: SyntaxWarning: invalid escape sequence '\-'
  if not all(c in '.,!?;:()[]{}|"\'/\-' for c in t["token"].strip())]
2026-03-28 17:19:55.246495: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774718395.325364    6849 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774718395.352247    6849 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774718395.444412    6849 computation_placer.cc:177] comp

In [ ]:
#@title Configure Cloudflared and Expose FastAPI
import subprocess
import time
import re

# Kill any existing cloudflared tunnels
!pkill -f cloudflared || true

# Start cloudflared tunnel in the background, log output to a file
cloudflared_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait and parse the public URL from cloudflared's output
public_url = None
print("Waiting for Cloudflare tunnel to establish...")

for _ in range(30):  # Try for ~15 seconds
    line = cloudflared_process.stdout.readline().decode("utf-8", errors="ignore")
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("✅ Public API URL:", public_url)
    print("📄 Swagger Docs:", public_url + "/docs")
else:
    print("❌ Could not detect public URL. Check cloudflared output manually.")

^C
Waiting for Cloudflare tunnel to establish...
✅ Public API URL: https://national-collecting-judicial-half.trycloudflare.com
📄 Swagger Docs: https://national-collecting-judicial-half.trycloudflare.com/docs


In [ ]:
#@title Test API via Cloudflared Tunnel (v2.0)
import requests, time, json

API_URL = public_url + "/predict"

email_text = "Dear user1, Our security systems have detected unusual login activity on your account.  To protect your account, you must verify your credentials within 24 hours  or your access will be temporarily suspended.  Please click the link below to confirm your identity:  https://phishing.test/verify  If you do not verify within 24 hours, your account will be locked pending a  security review.  IT Security Team  ABC Securities" #@param {type:"string"}
model_selector = "bert" #@param ["bert", "lstm", "gru"]
sender_input   = "" #@param {type:"string"}

payload = {
    "text":   email_text,
    "model":  model_selector,
    "sender": sender_input.strip() if sender_input.strip() else None,
}

try:
    time.sleep(3)
    response = requests.post(API_URL, json=payload)
    result   = response.json()

    SEP = "-" * 52
    print("\n" + "=" * 52)
    print(f"  PHISHING DETECTION  (model={result['model']})")
    print("=" * 52)
    print(f"  Result     : {result['prediction']}")
    print(f"  Confidence : {result['confidence']:.1%}")

    if result.get("url_analysis"):
        ua = result["url_analysis"]
        print(f"\n  {SEP}")
        print(f"  URL ANALYSIS  ({ua['url_count']} links found)")
        print(f"  Trusted: {ua['trusted_count']}  Suspicious: {ua['suspicious_count']}  Unknown: {ua['unknown_count']}")
        print(f"  URL Trust Score: {ua['url_trust_score']:.2f}")
        if ua["suspicious_urls"]:
            print(f"  Suspicious URLs: {ua['suspicious_urls']}")
        if ua["trusted_urls"]:
            print(f"  Trusted URLs  : {ua['trusted_urls']}")
        if ua["has_ip_url"]:
            print(f"  WARNING: IP-based URL detected!")
        if ua["has_url_shortener"]:
            print(f"  WARNING: URL shortener detected!")
        if ua["has_misleading_url"]:
            print(f"  WARNING: Misleading brand URL detected!")

    if result.get("domain_trust") and result["domain_trust"].get("adjustment_applied"):
        dt = result["domain_trust"]
        print(f"\n  {SEP}")
        print(f"  Domain: {dt['domain']}  Trust: {dt['trust_score']} ({dt['trust_label']})")
        print(f"  Adjusted: {dt['original_label']} @ {dt['original_confidence']:.1%}  -->  {result['prediction']} @ {result['confidence']:.1%}")

    if result.get("top_tokens"):
        print(f"\n  {SEP}")
        print(f"  TOP SHAP TOKENS")
        for t in result["top_tokens"]:
            bar = "#" * min(20, int(abs(t["shap_score"]) * 800))
            print(f"  {t['token']:<15} {t['shap_score']:+.4f}  {bar}")

    print("\n" + "=" * 52)
    print("\nFull JSON:\n", json.dumps(result, indent=2))

except requests.exceptions.ConnectionError as e:
    print(f"Connection error: {e}")
    print("Ensure FastAPI server and Cloudflare tunnel are running.")



  PHISHING DETECTION  (model=bert)
  Result     : Phishing
  Confidence : 87.0%

  ----------------------------------------------------
  URL ANALYSIS  (1 links found)
  Trusted: 0  Suspicious: 0  Unknown: 1
  URL Trust Score: 0.50

  ----------------------------------------------------
  TOP SHAP TOKENS
  user            +0.1344  ####################
  Dear            +0.1270  ####################
  1               +0.1219  ####################
  your            +0.0518  ####################
  you             +0.0507  ####################
  must            +0.0506  ####################
  https           +0.0392  ####################
  Securities      +0.0387  ####################
  protect         +0.0308  ####################
  Our             -0.0301  ####################
  Team            -0.0295  ####################
  or              +0.0294  ####################
  verify          +0.0287  ####################


Full JSON:
 {
  "model": "bert",
  "prediction": "Phishing",
  "con